# Enhanced Cleaning — Kompas
Remove inline cross-reference snippets: `Baca juga:...` yang tersebar di tengah artikel.

**Strategi:** Regex deteksi akhir judul artikel (Title Case) dan awal kalimat berikutnya
(huruf kapital diikuti minimal 4 huruf kecil + spasi + huruf kecil).

In [ ]:
import re
import pandas as pd

DATA_PATH   = "/kaggle/input/datasets/davinraffilio9/datalabeled/data_labeled"
OUTPUT_PATH = "/kaggle/working"

In [ ]:
df = pd.read_csv(f"{DATA_PATH}/kompas_labeled.csv")
print(f"Loaded: {df.shape}")
df.head(3)

## EDA — Before Cleaning

In [ ]:
print("=== Label Distribution ===")
print(df["label"].value_counts())
print(f"\nAvg content length: {df['content'].str.len().mean():.0f} chars")

count = df["content"].str.contains(r"Baca juga", case=False, na=False).sum()
print(f"Rows with 'Baca juga': {count} / {len(df)}")

occ = df["content"].str.count(r"(?i)Baca juga")
print(f"Avg occurrences per row (rows that have it): {occ[occ>0].mean():.2f}")
print(f"Max occurrences in one row               : {occ.max()}")

## Cleaning — Remove Inline `Baca juga:...`

Regex pattern:
```
(?i)baca\s+juga\s*:[^\n]*?(?=[A-Z][a-z]{3,}\s+[a-z]|\n|$)
```
- `(?i)` — case-insensitive
- `baca\s+juga\s*:` — match trigger phrase
- `[^\n]*?` — non-greedy: ambil karakter sampai lookahead match
- Lookahead `[A-Z][a-z]{3,}\s+[a-z]` — transisi ke kalimat baru (kapital + 4+ huruf kecil + spasi + huruf kecil)

In [ ]:
BACA_JUGA_RE = re.compile(
    r"(?i)baca\s+juga\s*:[^\n]*?(?=[A-Z][a-z]{3,}\s+[a-z]|\n|$)"
)

def remove_baca_juga(text: str) -> str:
    """Hapus semua snippet 'Baca juga:...' inline dalam artikel Kompas."""
    if pd.isna(text):
        return ""
    text = BACA_JUGA_RE.sub("", str(text))
    text = re.sub(r"[ \t]+", " ", text)       # collapse horizontal whitespace
    text = re.sub(r"\n{2,}", "\n", text)      # collapse blank lines
    return text.strip()

df["content_clean"] = df["content"].apply(remove_baca_juga)
affected = (df["content_clean"] != df["content"].fillna("")).sum()
print(f"Done. Rows affected: {affected} / {len(df)}")

## Before / After Comparison

In [ ]:
mask = df["content"].str.contains(r"Baca juga", case=False, na=False)
for _, row in df[mask].head(2).iterrows():
    raw   = str(row["content"])
    clean = str(row["content_clean"])
    idx   = raw.lower().find("baca juga")
    print("=== BEFORE (±150 chars around 'Baca juga') ===")
    print(raw[max(0,idx-50):idx+200])
    print("\n=== AFTER (same start position) ===")
    print(clean[max(0,idx-50):idx+150])
    print("-" * 70)

print(f"\nAvg length BEFORE: {df['content'].str.len().mean():.0f}")
print(f"Avg length AFTER : {df['content_clean'].str.len().mean():.0f}")

## Rebuild `text` = title + content_clean

In [ ]:
df["text"] = (
    df["title"].astype(str).str.strip() + ". " +
    df["content_clean"].astype(str).str.strip()
).str.strip()

df_out = df[["date", "title", "content_clean", "article_id", "text", "label"]].copy()
df_out = df_out.rename(columns={"content_clean": "content"})

print(f"Output shape : {df_out.shape}")
print(df_out["label"].value_counts())
df_out.head(3)

In [ ]:
df_out.to_csv(f"{OUTPUT_PATH}/kompas_labeled.csv", index=False, encoding="utf-8")
print(f"Saved: {OUTPUT_PATH}/kompas_labeled.csv  ({len(df_out)} rows)")